# 01. Pemahaman Masalah

## 1.1 Konteks Bisnis

Industri logistik dan pengiriman menangani volume pengiriman dalam jumlah besar yang memerlukan pemantauan berkelanjutan sepanjang proses pengiriman.

Dalam operasional logistik konvensional, status penanganan pengiriman sering kali diperbarui secara manual oleh staf operasional. Proses ini dapat menimbulkan beberapa tantangan:
- Keterlambatan pembaruan status.
- Pelaporan status pengiriman yang tidak akurat.
- Catatan operasional yang tidak konsisten.
- Kesulitan mengidentifikasi pengiriman yang berpotensi mengalami kendala pengiriman.

Tantangan-tantangan ini dapat memengaruhi efisiensi operasional dan kemampuan perusahaan logistik dalam memantau kondisi pengiriman.

## 1.2 Masalah Bisnis

Masalah utama yang dibahas dalam kompetisi ini adalah klasifikasi status penanganan pengiriman berdasarkan informasi pengiriman historis dan catatan operasional.

Tujuannya adalah mengembangkan model *machine learning* yang dapat memprediksi status penanganan setiap pengiriman secara otomatis menggunakan informasi yang tersedia seperti:
- Karakteristik pengiriman.
- Informasi pelanggan.
- Informasi kurir.
- Informasi pusat distribusi (*distribution hub*).
- Riwayat pelacakan pengiriman (*shipment tracking events*).
- Segmen rute.
- Catatan faktur (*invoice*) dan pembayaran.

Model diharapkan mampu mengidentifikasi status penanganan yang paling tepat untuk setiap pengiriman pada *test dataset*.

## 1.3 Tujuan Analisis

Tujuan utama dari analisis ini adalah:
1. Memahami karakteristik dari *dataset* pengiriman.
2. Mengidentifikasi faktor-faktor operasional relevan yang berkaitan dengan status penanganan pengiriman.
3. Menyelidiki hubungan antara catatan pengiriman dan tabel operasional pendukung.
4. Mengidentifikasi fitur-fitur prediktif yang berguna.
5. Mengembangkan model klasifikasi multikelas (*multiclass classification*).
6. Mengoptimalkan performa model menggunakan *Macro F1-score*.

Analisis ini akan memprioritaskan performa prediktif dengan tetap menjaga integritas data dan menghindari *target leakage*.

In [4]:
# ============================================================
# 1.2 PROBLEM FORMULATION
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from sklearn.metrics import f1_score, classification_report

# Competition information
COMPETITION_NAME = "Datavidia 11.0 - Task 1"

# Problem definition
PROBLEM_TYPE = "Supervised Multiclass Classification"

TARGET = "status_penanganan"
ID_COL = "shipment_id"

# Target classes
TARGET_CLASSES = [
    "normal",
    "konfirmasi_alamat",
    "tertunda_operasional",
    "gagal_antar_awal",
    "retur_pengirim",
    "rusak_perjalanan",
    "klaim_hilang"
]

N_CLASSES = len(TARGET_CLASSES)

# Evaluation metric
EVALUATION_METRIC = "Macro F1-Score"

problem_info = {
    "Competition": COMPETITION_NAME,
    "Problem Type": PROBLEM_TYPE,
    "Target Variable": TARGET,
    "Identifier": ID_COL,
    "Number of Classes": N_CLASSES,
    "Evaluation Metric": EVALUATION_METRIC
}

display(
    pd.DataFrame(
        problem_info.items(),
        columns=["Attribute", "Description"]
    )
)

,Attribute,Description
0,Competition,Datavidia 11.0 - Task 1
1,Problem Type,Supervised Multiclass Classification
2,Target Variable,status_penanganan
3,Identifier,shipment_id
4,Number of Classes,7
5,Evaluation Metric,Macro F1-Score


## 1.3 Definisi Variabel Target

Variabel target untuk kompetisi ini adalah `status_penanganan`.

Variabel ini merepresentasikan status penanganan yang ditetapkan untuk setiap pengiriman.

Tugas ini melibatkan prediksi salah satu dari tujuh kelas yang tersedia.

| Kelas | Deskripsi |
|---|---|
| normal | Pengiriman selesai tanpa kendala |
| konfirmasi_alamat | Diperlukan konfirmasi atau perbaikan alamat |
| tertunda_operasional | Pengiriman tertunda karena masalah operasional |
| gagal_antar_awal | Percobaan pengiriman awal gagal |
| retur_pengirim | Pengiriman dikembalikan ke pengirim |
| rusak_perjalanan | Pengiriman rusak selama proses transportasi |
| klaim_hilang | Pengiriman hilang dan terkait dengan pengajuan klaim |

Setiap pengiriman harus ditetapkan ke tepat satu kelas target.

Model klasifikasi harus mempelajari hubungan antara karakteristik pengiriman, perilaku pelanggan, performa kurir, aktivitas operasional, dan status penanganan yang sesuai.

In [ ]:
target_descriptions = {
    "normal": "Shipment completed without incidents",
    "konfirmasi_alamat": "Address confirmation or correction required",
    "tertunda_operasional": "Operational delivery delay",
    "gagal_antar_awal": "Failed initial delivery attempt",
    "retur_pengirim": "Shipment returned to sender",
    "rusak_perjalanan": "Shipment damaged during transportation",
    "klaim_hilang": "Shipment lost and associated with a claim"
}

target_df = pd.DataFrame({
    "Class": TARGET_CLASSES,
    "Description": [
        target_descriptions[c]
        for c in TARGET_CLASSES
    ]
})

display(target_df)

print(f"Total target classes: {N_CLASSES}")

,Class,Description
0,normal,Shipment completed without incidents
1,konfirmasi_alamat,Address confirmation or correction required
2,tertunda_operasional,Operational delivery delay
3,gagal_antar_awal,Failed initial delivery attempt
4,retur_pengirim,Shipment returned to sender
5,rusak_perjalanan,Shipment damaged during transportation
6,klaim_hilang,Shipment lost and associated with a claim


Total target classes: 7


## 1.4 Struktur Dataset

Dataset kompetisi ini terdiri dari sembilan tabel relasional.

Tabel-tabel ini berisi informasi pengiriman serta catatan operasional pendukung.

Dataset dapat dibagi menjadi tiga kategori utama:

### A. Data Transaksi Utama

- shipments

Berisi catatan pengiriman utama dan variabel target pada *training dataset*.

### B. Data Transaksi Operasional

- scan_events
- route_segments
- invoices

Tabel-tabel ini berisi beberapa catatan operasional yang terkait dengan masing-masing pengiriman.

### C. Data Master

- customers
- couriers
- hubs
- service_packages
- promotions

Tabel-tabel ini menyediakan informasi tambahan mengenai entitas yang terkait dengan setiap pengiriman.

### Struktur Relasional

`shipment_id` berperan sebagai pengenal utama (*primary identifier*) untuk analisis di tingkat pengiriman.

Tabel operasional pendukung dapat berisi beberapa catatan untuk satu pengiriman. Oleh karena itu, diperlukan agregasi yang tepat sebelum menggabungkannya (*joining*) dengan tabel pengiriman utama.

Hal ini penting untuk mencegah duplikasi observasi selama proses *feature engineering*.

In [6]:
dataset_structure = [
    {
        "Table": "shipments",
        "Category": "Main Transaction",
        "Relationship": "One row per shipment",
        "Primary Key": "shipment_id",
        "Purpose": "Main shipment characteristics and target"
    },
    {
        "Table": "scan_events",
        "Category": "Operational Transaction",
        "Relationship": "One-to-Many",
        "Primary Key": "shipment_id + event identifier",
        "Purpose": "Shipment tracking and scanning activities"
    },
    {
        "Table": "route_segments",
        "Category": "Operational Transaction",
        "Relationship": "One-to-Many",
        "Primary Key": "shipment_id + urutan",
        "Purpose": "Shipment transportation routes"
    },
    {
        "Table": "invoices",
        "Category": "Operational Transaction",
        "Relationship": "One-to-Many",
        "Primary Key": "Invoice identifier",
        "Purpose": "Payment and invoice information"
    },
    {
        "Table": "customers",
        "Category": "Master Data",
        "Relationship": "Many-to-One",
        "Primary Key": "customer_id",
        "Purpose": "Customer characteristics"
    },
    {
        "Table": "couriers",
        "Category": "Master Data",
        "Relationship": "Many-to-One",
        "Primary Key": "courier_id",
        "Purpose": "Courier characteristics"
    },
    {
        "Table": "hubs",
        "Category": "Master Data",
        "Relationship": "Many-to-One",
        "Primary Key": "hub_id",
        "Purpose": "Distribution hub characteristics"
    },
    {
        "Table": "service_packages",
        "Category": "Master Data",
        "Relationship": "Many-to-One",
        "Primary Key": "package_id",
        "Purpose": "Shipment service characteristics"
    },
    {
        "Table": "promotions",
        "Category": "Master Data",
        "Relationship": "Many-to-One",
        "Primary Key": "promo_id",
        "Purpose": "Promotional information"
    }
]

dataset_structure_df = pd.DataFrame(dataset_structure)

display(dataset_structure_df)

,Table,Category,Relationship,Primary Key,Purpose
0,shipments,Main Transaction,One row per shipment,shipment_id,Main shipment characteristics and target
1,scan_events,Operational Transaction,One-to-Many,shipment_id + event identifier,Shipment tracking and scanning activities
2,route_segments,Operational Transaction,One-to-Many,shipment_id + urutan,Shipment transportation routes
3,invoices,Operational Transaction,One-to-Many,Invoice identifier,Payment and invoice information
4,customers,Master Data,Many-to-One,customer_id,Customer characteristics
5,couriers,Master Data,Many-to-One,courier_id,Courier characteristics
6,hubs,Master Data,Many-to-One,hub_id,Distribution hub characteristics
7,service_packages,Master Data,Many-to-One,package_id,Shipment service characteristics
8,promotions,Master Data,Many-to-One,promo_id,Promotional information


## 1.5 Metrik Evaluasi

Kompetisi ini menggunakan *Macro F1-score* sebagai metrik evaluasi utama.

*Macro F1* menghitung *F1-score* secara independen untuk setiap kelas
dan kemudian menghitung rata-rata tanpa bobot (*unweighted average*) di seluruh kelas.

Rumusnya adalah:

Macro F1 = (1/K) × Σ F1(k)

di mana:

- K merepresentasikan total jumlah kelas target.
- F1(k) merepresentasikan *F1-score* dari kelas k.

*F1-score* untuk setiap kelas dihitung menggunakan:

F1 = 2 × (Precision × Recall) / (Precision + Recall)

Berbeda dengan akurasi, *Macro F1* memberikan tingkat kepentingan yang sama pada setiap kelas,
tanpa memandang jumlah observasinya.

Oleh karena itu, model harus mempertahankan performa prediktif yang seimbang
di seluruh tujuh status penanganan pengiriman.

Metrik ini membuat klasifikasi kelas minoritas menjadi sangat penting.

In [7]:
def macro_f1_score(y_true, y_pred):
    """
    Calculate Macro F1-score across all target classes.
    """

    unknown_labels = (
        set(y_pred) - set(TARGET_CLASSES)
    )

    if unknown_labels:
        raise ValueError(
            f"Unknown predicted labels: {unknown_labels}"
        )

    return f1_score(
        y_true,
        y_pred,
        labels=TARGET_CLASSES,
        average="macro",
        zero_division=0
    )


print("Evaluation Metric:", EVALUATION_METRIC)
print("Number of Classes:", N_CLASSES)
print("Averaging Method: Macro")

Evaluation Metric: Macro F1-Score
Number of Classes: 7
Averaging Method: Macro


In [10]:
import os
from pathlib import Path

print("Direktori saat ini:", Path.cwd())
print("Isi direktori saat ini:", os.listdir("."))

Direktori saat ini: c:\Users\ASUS\.vscode\arkavidia
Isi direktori saat ini: ['01_datavidia_task1_eda.ipynb', '01_eda.py', '02_features.py', '03_train.py', '04_predict.py', 'cat_cols.json', 'data', 'feats.parquet', 'models']


In [18]:
def find_data_dir():

    candidates = []

    if os.getenv("DATA_DIR"):
        candidates.append(Path(os.environ["DATA_DIR"]))

    candidates.extend([
        Path("data"),
        Path("/kaggle/input/task-1-datavidia-11-0"),
        Path(
            "/kaggle/input/task-1-datavidia-11-0/"
            "Task-1-Datavidia-11.0"
        )
    ])

    for path in candidates:

        if (
            (path / "train" / "shipments.csv").exists()
            and (path / "test" / "shipments.csv").exists()
            and (path / "sample_submission.csv").exists()
        ):
            return path

    raise FileNotFoundError(
        "Dataset directory not found. "
        "Please configure DATA_DIR."
    )

DATA_DIR = Path(r"c:\Users\ASUS\.vscode\arkavidia")

TRAIN_PATH = DATA_DIR / "data" / "Task-1-Datavidia-11.0" / "train" / "shipments.csv"
TEST_PATH = DATA_DIR / "data" / "Task-1-Datavidia-11.0" / "test" / "shipments.csv"
SUBMISSION_PATH = DATA_DIR / "data" / "Task-1-Datavidia-11.0" / "sample_submission.csv"

print("Dataset directory:", DATA_DIR.resolve())
print("Train file:", TRAIN_PATH)
print("Test file:", TEST_PATH)
print("Sample submission:", SUBMISSION_PATH)

Dataset directory: C:\Users\ASUS\.vscode\arkavidia
Train file: c:\Users\ASUS\.vscode\arkavidia\data\Task-1-Datavidia-11.0\train\shipments.csv
Test file: c:\Users\ASUS\.vscode\arkavidia\data\Task-1-Datavidia-11.0\test\shipments.csv
Sample submission: c:\Users\ASUS\.vscode\arkavidia\data\Task-1-Datavidia-11.0\sample_submission.csv


In [19]:
train_key = pl.read_csv(
    TRAIN_PATH,
    columns=[ID_COL, TARGET]
)

test_key = pl.read_csv(
    TEST_PATH,
    columns=[ID_COL]
)

sample_submission = pl.read_csv(
    SUBMISSION_PATH
)

# Basic information
n_train = train_key.height
n_test = test_key.height

observed_classes = set(
    train_key[TARGET]
    .drop_nulls()
    .unique()
    .to_list()
)

missing_classes = (
    set(TARGET_CLASSES) - observed_classes
)

unknown_classes = (
    observed_classes - set(TARGET_CLASSES)
)

# Validation checks
validation_results = {
    "Target exists in train": TARGET in train_key.columns,
    "Target absent from test": TARGET not in test_key.columns,
    "All expected classes exist": len(missing_classes) == 0,
    "No unknown target classes": len(unknown_classes) == 0,
    "Train shipment ID unique": (
        train_key[ID_COL].n_unique() == n_train
    ),
    "Test shipment ID unique": (
        test_key[ID_COL].n_unique() == n_test
    ),
    "Submission ID matches test": (
        sample_submission.height == n_test
        and ID_COL in sample_submission.columns
        and sample_submission[ID_COL].equals(
            test_key[ID_COL]
        )
    )
}

validation_df = pd.DataFrame(
    [
        {
            "Validation": key,
            "Result": value
        }
        for key, value in validation_results.items()
    ]
)

display(validation_df)

print("\nDataset Summary")
print("----------------------")
print(f"Training shipments : {n_train:,}")
print(f"Testing shipments  : {n_test:,}")
print(f"Target classes     : {len(observed_classes)}")

if missing_classes:
    print("Missing classes:", missing_classes)

if unknown_classes:
    print("Unknown classes:", unknown_classes)

,Validation,Result
0,Target exists in train,True
1,Target absent from test,True
2,All expected classes exist,True
3,No unknown target classes,True
4,Train shipment ID unique,True
5,Test shipment ID unique,True
6,Submission ID matches test,True



Dataset Summary
----------------------
Training shipments : 1,243,617
Testing shipments  : 362,245
Target classes     : 7
